In [1]:
import json
import syncedlyrics
from mutagen.flac import FLAC
from mutagen.mp4 import MP4
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

MP4_TAG_MAP = {"title": "©nam", "artist": "©ART", "album": "©alb"}

ModuleNotFoundError: No module named 'syncedlyrics'

In [ ]:
def get_flac_metadata(filepath: str) -> dict:
    audio = FLAC(filepath)
    return {k: (audio.get(k) or [""])[0].strip() for k in ("title", "artist", "album")}

def get_alac_metadata(filepath: str) -> dict:
    audio = MP4(filepath)
    return {
        k: (audio.tags.get(tag) or [""])[0].strip()
        for k, tag in MP4_TAG_MAP.items()
    }

def get_metadata(filepath: Path) -> dict | None:
    try:
        if filepath.suffix.lower() == ".flac":
            return get_flac_metadata(str(filepath))
        elif filepath.suffix.lower() == ".m4a":
            return get_alac_metadata(str(filepath))
    except Exception as e:
        print(f"  Ошибка метаданных {filepath.name}: {e}")
    return None

def get_lyrics(title: str, artist: str) -> str | None:
    try:
        return syncedlyrics.search(
            f"{title} {artist}",
            providers=["Lrclib", "NetEase", "Megalobiz"],
            plain_only = True
        )
    except Exception:
        return None

def process_file(filepath: Path) -> dict | None:
    meta = get_metadata(filepath)
    if not meta or not meta["title"] or not meta["artist"]:
        print(f"  Пропуск: {filepath.name}")
        return None

    lyrics = get_lyrics(meta["title"], meta["artist"])
    print(f"{'✓' if lyrics else '✗'} {meta['artist']} — {meta['title']}")
    return {**meta, "lyrics": lyrics}

def fetch_lyrics_bulk(folder: str, output_file: str = "lyrics.json", workers: int = 8):
    audio_files = [
        p for p in Path(folder).rglob("*")
        if p.suffix.lower() in (".flac", ".m4a")
    ]
    print(f"Найдено файлов: {len(audio_files)}, воркеров: {workers}")

    results = {}
    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = {executor.submit(process_file, f): f for f in audio_files}
        for future in as_completed(futures):
            meta = future.result()
            if meta:
                key = f"{meta['artist']} — {meta['title']}"
                results.setdefault(key, meta)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    found = sum(1 for v in results.values() if v["lyrics"])
    print(f"\nГотово: {found}/{len(results)} текстов найдено → {output_file}")
    return results



In [ ]:
folder = "S:\Music"
fetch_lyrics_bulk(folder=folder)

In [ ]:
[(k, v.get('lyrics').splitlines()) for k, v in data.items()]

In [2]:
import json
import re

In [15]:
with open('lyrics.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

In [17]:
song_text = list()
for k, v in data.items():
    song_text_record = {
        'title':v.get('title', 'N/A'), 
        'artist':v.get('artist', 'N/A'), 
        'album':v.get('album', 'N/A'),
        'lyrics':v.get('lyrics')
        }
    if song_text_record.get('lyrics'):
        song_text.append(song_text_record)
song_text = tuple(song_text)

In [18]:
for item in song_text:
    lyrics = item.get('lyrics')
    item['lyrics'] = re.sub(r'\[.*?\]', '', lyrics).strip()

In [38]:
song_text[50]

{'title': 'The Man’s Too Strong',
 'artist': 'Dire Straits',
 'album': 'Brothers in Arms',
 'lyrics': "I am just an ageing drummer boy\n And in the wars I used to play\n And I've called a tune to many a torture session\n Now they say I am a war criminal\n And I'm fading away\n Father, please hear my confession\n I have legalized robbery\n Called it belief\n I have run with the money\n I have a-hid like a thief\n Re-written history\n With my armies and my crooks\n Invented memories\n I did burn all the books\n And I can still hear his laughter\n And I can still hear his song\n The man's too big\n The man's too strong\n Well I tried to be meek\n I have tried to be mild\n But I spat like a woman\n And I sulked like a child\n I have lived behind walls\n That have made me alone\n Striven for peace\n Which I never have known\n And I can still hear his laughter\n And I can still hear his song\n The man's too big\n The man's too strong\n Well the sun rose on the courtyard\n And they all did he

In [3]:
!docker run -d -p 6333:6333 -p 6334:6334 \
   -v "./qdrant_storage:/qdrant/storage:z" \
   qdrant/qdrant

66539d4dd2648d0dfd2a9aa10d12b1c42b729958996c510dfb60c95cd4a4fbd7


In [4]:
from qdrant_client import QdrantClient
from qdrant_client import models

client = QdrantClient("http://localhost:6333")
client.get_collections()

CollectionsResponse(collections=[CollectionDescription(name='lyrics-sparse-and-dense'), CollectionDescription(name='lyrics-sparse')])

In [43]:
from qdrant_client import models

# Create the collection with specified sparse vector parameters
client.create_collection(
    collection_name="lyrics-sparse",
    sparse_vectors_config={
        "bm25": models.SparseVectorParams(
            modifier=models.Modifier.IDF,
        )
    }
)

True

In [ ]:
import uuid

# Send the points to the collection
client.upsert(
    collection_name="lyrics-sparse",
    points=[
        models.PointStruct(
            id=uuid.uuid4().hex,
            vector={
                "bm25": models.Document(
                    text=song["lyrics"], 
                    model="Qdrant/bm25",
                ),
            },
            payload={
                "lyrics": song["lyrics"],
                "title": song["title"],
                "artist": song["artist"],
                "album": song["album"],
            }
        )
        for song in song_text
    ]
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [9]:
def search(query: str, limit: int = 1) -> list[models.ScoredPoint]:
    results = client.query_points(
        collection_name="lyrics-sparse",
        query=models.Document(
            text=query,
            model="Qdrant/bm25",
        ),
        using="bm25",
        limit=limit,
        with_payload=True,
    )

    return results.points

In [10]:
results = search("Mercedes", limit = 5)
print(f"Найдено {len(results)} результатов")
if results:
    print(f"Самый релевантный результат: {results[0].payload['artist']} - {results[0].payload['title']}")

Найдено 5 результатов
Самый релевантный результат: Puff Daddy & the Family - Been Around the World


Теперь добавляем dense вектора

In [14]:
client.delete_collection("lyrics-sparse-and-dense")

True

In [19]:
client.create_collection(
    collection_name="lyrics-sparse-and-dense",
    vectors_config={
        # Named dense vector for jinaai/jina-embeddings-v2-small-en
        "jina-small": models.VectorParams(
            size=512,
            distance=models.Distance.COSINE,
        ),
    },
    sparse_vectors_config={
        "bm25": models.SparseVectorParams(
            modifier=models.Modifier.IDF,
        )
    }
)

True

In [20]:
def upsert_in_batches(client, collection_name, points, batch_size=100):
    for i in range(0, len(points), batch_size):
        batch = points[i:i + batch_size]
        client.upsert(
            collection_name=collection_name,
            points=batch
        )
        print(f"Upserted {min(i + batch_size, len(points))}/{len(points)}")

In [21]:
import uuid

In [22]:
upsert_in_batches(
    client, 
    collection_name="lyrics-sparse-and-dense",
    points=[
        models.PointStruct(
            id=uuid.uuid4().hex,
            vector={
                "jina-small": models.Document(
                    text=song["lyrics"],
                    model="jinaai/jina-embeddings-v2-small-en",
                ),
                "bm25": models.Document(
                    text=song["lyrics"], 
                    model="Qdrant/bm25",
                ),
            },
            payload={
                "lyrics": song["lyrics"],
                "title": song["title"],
                "artist": song["artist"],
                "album": song["album"],
            }
        )
        for song in song_text
    ]
    )

Upserted 100/5441
Upserted 200/5441
Upserted 300/5441
Upserted 400/5441
Upserted 500/5441
Upserted 600/5441
Upserted 700/5441
Upserted 800/5441
Upserted 900/5441
Upserted 1000/5441
Upserted 1100/5441
Upserted 1200/5441
Upserted 1300/5441
Upserted 1400/5441
Upserted 1500/5441
Upserted 1600/5441
Upserted 1700/5441
Upserted 1800/5441
Upserted 1900/5441
Upserted 2000/5441
Upserted 2100/5441
Upserted 2200/5441
Upserted 2300/5441
Upserted 2400/5441
Upserted 2500/5441
Upserted 2600/5441
Upserted 2700/5441
Upserted 2800/5441
Upserted 2900/5441
Upserted 3000/5441
Upserted 3100/5441
Upserted 3200/5441
Upserted 3300/5441
Upserted 3400/5441
Upserted 3500/5441
Upserted 3600/5441
Upserted 3700/5441
Upserted 3800/5441
Upserted 3900/5441
Upserted 4000/5441
Upserted 4100/5441
Upserted 4200/5441
Upserted 4300/5441
Upserted 4400/5441
Upserted 4500/5441
Upserted 4600/5441
Upserted 4700/5441
Upserted 4800/5441
Upserted 4900/5441
Upserted 5000/5441
Upserted 5100/5441
Upserted 5200/5441
Upserted 5300/5441
Up

In [23]:
def multi_stage_search(query: str, limit: int = 1) -> list[models.ScoredPoint]:
    results = client.query_points(
        collection_name="lyrics-sparse-and-dense",
        prefetch=[
            models.Prefetch(
                query=models.Document(
                    text=query,
                    model="jinaai/jina-embeddings-v2-small-en",
                ),
                using="jina-small",
                # Prefetch ten times more results, then
                # expected to return, so we can really rerank
                limit=(10 * limit),
            ),
        ],
        query=models.Document(
            text=query,
            model="Qdrant/bm25", 
        ),
        using="bm25",
        limit=limit,
        with_payload=True,
    )

    return results.points

In [77]:
results = multi_stage_search("ferrari", limit = 5)
print(f"Найдено {len(results)} результатов")
if results:
    print(f"Самый релевантный результат: {results[0].payload['artist']} - {results[0].payload['title']}")

Найдено 4 результатов
Самый релевантный результат: Roddy Ricch - Underdog


In [83]:
results[0].payload

{'lyrics': 'Slide out the garage (Roof)\n I got a hundred cars\n This for the underdogs\n The teller just asked me how I want the cash in\n I told him, "A hundred, large"\n I want them gold wraps\n Printed up, double R\n Big dog, big dog, big dog, roof\n Poppin\' out, sayin\' I ain\'t great, who?\n Biscayne drive, I cut off my roof\n You know how I\'m comin\', it\'s take two\n Tell a-, "Get out the way"\n Four by four, big Phantom (Shoo)\n They thought I was cancelled, I piped up and flipped the channel\n LST on the brim, they ain\'t think we\'d win, I just talked to my brother (Woo)\n Ain\'t safe when you- with the underdog, I might bite, this get rough (Roof)\n I\'m tryna have paper, Robert Craft and put a Tom Brady\n On a stab, pull up on Dre, talking-, he- with the wave\n Straight out of Compton\n Slide out the garage (Roof)\n I got a hundred cars\n This for the underdogs\n The teller just asked me how I want the cash in\n I told him, "A hundred, large"\n I want them gold wraps\n P

In [17]:
results[0].payload['title']

'Breaking News (Skit)'

In [25]:
def rrf_search(query: str, limit: int = 1) -> list[models.ScoredPoint]:
    results = client.query_points(
        collection_name="lyrics-sparse-and-dense",
        prefetch=[
            models.Prefetch(
                query=models.Document(
                    text=query,
                    model="jinaai/jina-embeddings-v2-small-en",
                ),
                using="jina-small",
                limit=(5 * limit),
            ),
            models.Prefetch(
                query=models.Document(
                    text=query,
                    model="Qdrant/bm25",
                ),
                using="bm25",
                limit=(5 * limit),
            ),
        ],
        # Fusion query enables fusion on the prefetched results
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        with_payload=True,
    )

    return results.points

In [84]:
result = rrf_search("ferrari")

In [90]:
result[5].payload

{'lyrics': "Yeah\n Getting it in from here to china\n Yo\n Alright let me just cool this right now quickly\n Super trippy riding through the gritty inner city\n Roll with the committee\n Handle your business or handle the pity\n All I see is lots of titties\n I know bunch of hippie chicks\n That's ready to show me tricks\n They doing the splits\n I'm all up in the mix\n A choir out the mist\n I'm taking trips\n I'm in the Ferrari looking sick\n I'm in the Ferrari looking slick\n Letting the engine rip\n And getting a tire\n A' grippin' ah slippin' and slidin'\n Turning the music up\n I'm vibing now I'm flying\n Lord strap me if I'm lying\n I ain't perfect but I'm trying\n Going super sire and buying\n Anything that catches my eye\n Cause I'm a provider getting it in from here to china\n It's so minor I'm a survivor, never retire\n I'm a black tiger ready to blaze to the fire, live wire\n Now I'm rolling through the shires\n Blazing the green to get me higher\n Now I'm inspired\n Puttin